#### **BUILD**

In [3]:
import logging
import os
try:
    from pathlib2 import Path
except ImportError:
    from pathlib import Path

import numpy as np
import torch
import torch.multiprocessing as mp
import parmap

import sys
sys.path.insert(0, '/u/gongt/spikes_localization_registration')

from detect.detector import Detect
from localization_pipeline.denoiser import Denoise

from detect.deduplication import deduplicate_gpu, deduplicate

from scipy.signal import argrelmin

from detect.run import run

In [4]:
import json

REPO = '/u/gongt/spikes_localization_registration'
REC_DIR = '/ptmp/gongt/GLC7189/GLC7189_g_g1/preprocessed'
PEAKS_DIR = REC_DIR + '/thresholding_peaks_3.3'

# ── geometry: build from our probe.json (368 ch, x=[0,16,32,48], y=0-3820 µm) ──
# The repo's np2_channel_map.npy uses different geometry (384 ch, 15 µm pitch) -- not ours
with open(REC_DIR + '/probe.json') as f:
    _probe = json.load(f)
geom_array = np.array(_probe['probes'][0]['contact_positions'])  # (368, 2)
print(f"geom_array shape: {geom_array.shape}  x_unique: {np.unique(geom_array[:,0])}  y: {geom_array[:,1].min():.0f}-{geom_array[:,1].max():.0f} µm")

path_nn_denoiser = REPO + '/pretrained_denoiser/denoise.pt'
standardized_path = REC_DIR + '/traces_cached_seg0.raw'
standardized_dtype = 'float16'
n_channels = geom_array.shape[0]   # 368
sampling_rate = 30000
len_recording = 36075              # seconds (≈ 601 min)

# denoiser / detection params (detection section is skipped — we use our own peaks)
spike_size_nn = 121
n_filters_denoise = [16, 8, 4]
filter_sizes_denoise = [5, 11, 21]
n_batches = len_recording          # one 1-second batch per second


geom_array shape: (368, 2)  x_unique: [ 0. 16. 32. 48.]  y: 0-3820 µm


In [5]:
# ── DETECTION SKIPPED ────────────────────────────────────────────────────────
# We use peaks_combined2_sorted.npy from SpikeInterface thresholding (126M peaks).
# Boussard's run() detector is not needed.
# run(...)


#### **LOCALIZATION**

In [6]:
import os
import numpy as np
from tqdm import tqdm
# from residual import RESIDUAL
from localization_pipeline.localizer import LOCALIZER
from localization_pipeline.merge_results import get_merged_arrays

In [7]:
import json, numpy as np, os

REPO      = '/u/gongt/spikes_localization_registration'
REC_DIR   = '/ptmp/gongt/GLC7189/GLC7189_g_g1/preprocessed'
PEAKS_DIR = REC_DIR + '/thresholding_peaks_3.3'
OUT_DIR   = PEAKS_DIR + '/boussard_localization'

bin_file       = REC_DIR + '/traces_cached_seg0.raw'
dtype_input    = 'float16'
n_channels     = 368
sampling_rate  = 30000

denoiser_weights = REPO + '/pretrained_denoiser/denoise.pt'

# ── geometry: 368-channel post-preprocessed map (probe.json) ─────────────────
# We must use the 368-ch map, not the raw 384-ch map.
# Reason: traces_cached_seg0.raw has exactly 368 channels; read_waveforms()
# computes byte offsets as sample_index * n_channels * itemsize, so n_channels
# must match the binary. probe.json stores the correct physical (x,y) positions
# of the 368 kept channels in preprocessed channel order.
# The 16 removed channels (scattered across the probe) create small spatial gaps
# but have no effect on localization quality.
with open(REC_DIR + '/probe.json') as f:
    _probe = json.load(f)
geom_array = np.array(_probe['probes'][0]['contact_positions'])  # (368, 2)
print(f"geom_array: {geom_array.shape}  x={np.unique(geom_array[:,0])}  y={geom_array[:,1].min():.0f}-{geom_array[:,1].max():.0f} µm")

# ── spike index: [sample_index, channel_index], shape (N, 2) ─────────────────
_peaks = np.load(PEAKS_DIR + '/peaks_combined2_sorted.npy')
spike_index_all = np.stack([_peaks['sample_index'], _peaks['channel_index']], axis=1)
spike_index_all = spike_index_all[spike_index_all[:, 0].argsort()]

# ── limit to first 30 min (0 – 1800 s) ───────────────────────────────────────
len_recording = 1800   # seconds
n_batches     = len_recording
MAX_SAMPLE    = len_recording * sampling_rate  # 54_000_000

spike_index = spike_index_all[spike_index_all[:, 0] < MAX_SAMPLE]
print(f"All peaks: {len(spike_index_all):,}  →  30-min subset: {len(spike_index):,}")

fname_spike_train = PEAKS_DIR + '/boussard_spike_index_30min.npy'
np.save(fname_spike_train, spike_index)
print(f"Saved: {fname_spike_train}")

# ── denoiser alignment offset ─────────────────────────────────────────────────
# read_waveforms() treats its input as the window CENTER (reads ±60 samples).
# Denoiser expects the trough at position 42 from the window start.
# If center = trough + K, trough position in window = 60 - K.
# To get trough at 42: K = 60 - 42 = 18.
# Our sample_index IS the trough sample, same as their YASS-style detector.
denoiser_min = 42
detector_min = 60   # = n_times//2; trough at window center → position 60
offset_detector_denoiser = detector_min - denoiser_min  # = 18
print(f"offset_detector_denoiser: {offset_detector_denoiser}  (trough will land at sample {60 - offset_detector_denoiser} in the denoiser window)")

fname_templates = None
n_filters_denoise    = [16, 8, 4]
filter_sizes_denoise = [5, 11, 21]


geom_array: (368, 2)  x=[ 0. 16. 32. 48.]  y=0-3820 µm
All peaks: 126,258,007  →  30-min subset: 7,225,769
Saved: /ptmp/gongt/GLC7189/GLC7189_g_g1/preprocessed/thresholding_peaks_3.3/boussard_spike_index_30min.npy
offset_detector_denoiser: 18  (trough will land at sample 42 in the denoiser window)


In [8]:
localizer_obj = LOCALIZER(
    bin_file,
    dtype_input,
    fname_spike_train,
    fname_templates,
    geom_array,           # array accepted directly (no need to save to disk)
    denoiser_weights,
    offset_detector_denoiser,
    n_filters=n_filters_denoise,
    filter_sizes=filter_sizes_denoise,
    sampling_rate=sampling_rate,
)
localizer_obj.load_denoiser()


RuntimeError: No CUDA GPUs are available

#### **OFFSET ALIGNMENT CHECK** — verify trough lands at position 42

In [ ]:
# ── Trough alignment check ────────────────────────────────────────────────────
# read_waveforms(center) returns a 121-sample window where window[60] = center.
# We pass center = trough_sample + offset_detector_denoiser  (= trough + 18)
# → trough lands at position  60 - 18 = 42 in the window.
# This cell reads 20 random batch-0 spikes at their main channel and overlays
# them. If the offset is correct every minimum should fall on the dashed line.
# Traces are coloured by PTP; values are printed on the right margin.

import matplotlib.pyplot as plt
import matplotlib.cm as cm

PLOTS_DIR = PEAKS_DIR + '/boussard_plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

N_ALIGN = 20
_rng = np.random.default_rng(42)

_b0_mask  = spike_index[:, 0] < sampling_rate
_b0_times = spike_index[_b0_mask, 0]
_b0_chans = spike_index[_b0_mask, 1]
_sel      = _rng.choice(len(_b0_times), size=min(N_ALIGN, len(_b0_times)), replace=False)

_traces, _ptps, _trough_at = [], [], []
for i in _sel:
    center = int(_b0_times[i] + offset_detector_denoiser)
    wf, skip = localizer_obj.read_waveforms(np.array([center]))   # (1, 121, 368)
    if len(skip) > 0:
        continue                         # near boundary — skip
    mc = int(_b0_chans[i])
    trace = wf[0, :, mc].copy()         # (121,) raw at detected main channel
    _traces.append(trace)
    _ptps.append(float(trace.ptp()))
    _trough_at.append(int(trace.argmin()))

_tr   = np.array(_traces)               # (N, 121)
_ptp  = np.array(_ptps)                 # (N,)
_pn   = (_ptp - _ptp.min()) / (_ptp.max() - _ptp.min() + 1e-12)   # → [0, 1]
_cmap = cm.get_cmap('plasma')
_tax  = np.arange(121)

fig, ax = plt.subplots(figsize=(10, 5))

for trace, pn, ptp_val in zip(_tr, _pn, _ptp):
    c = _cmap(pn)
    ax.plot(_tax, trace, color=c, lw=0.9, alpha=0.85)
    ax.text(122, trace[-1], f'{ptp_val:.3f}', color=c, fontsize=6, va='center')
ax.axvline(42, color='k', lw=1.5, ls='--', label='expected trough (pos 42)')
ax.set_xlabel('Sample in window (0 – 120)', fontsize=12)
ax.set_ylabel('Amplitude (raw)', fontsize=12)
ax.set_title(f'Trough alignment check  (offset_detector_denoiser = {offset_detector_denoiser})\n'
             f'Raw waveforms at main channel  (N = {len(_tr)})  |  PTP labels on right',
             fontsize=11)
ax.legend(fontsize=9)
_sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(_ptp.min(), _ptp.max()))
_sm.set_array([])
plt.colorbar(_sm, ax=ax, label='PTP amplitude')

plt.tight_layout()
plt.savefig(PLOTS_DIR + '/alignment_check.png', dpi=150, bbox_inches='tight')
plt.show()

_ta = np.array(_trough_at)
print(f"Actual trough positions: {sorted(_ta.tolist())}")
print(f"Mean ± std:  {_ta.mean():.1f} ± {_ta.std():.1f}   (target = 42)")
print(f"All at 42?   {'YES ✓' if (_ta == 42).all() else 'NO ✗ — adjust offset_detector_denoiser'}")
print(f"Saved → {PLOTS_DIR}/alignment_check.png")


#### **RUN LOCALIZATION** — 1800 × 1-second batches (first 30 min)

In [ ]:

os.makedirs(OUT_DIR, exist_ok=True)
for i in tqdm(range(n_batches)):
    localizer_obj.get_estimate(i, threshold=6, output_directory=OUT_DIR)


#### **DENOISER QUALITY CHECK** — raw vs denoised waveforms on batch 0

In [ ]:

import matplotlib.pyplot as plt
import torch

PLOTS_DIR = PEAKS_DIR + '/boussard_plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

# Pick spikes from batch 0 (first 1 second)
_b0_mask   = (spike_index[:, 0] < sampling_rate)
_b0_times  = spike_index[_b0_mask, 0]
_b0_chans  = spike_index[_b0_mask, 1]
print(f"Batch 0: {len(_b0_times)} spikes")

N_SHOW = 5
rng = np.random.default_rng(42)
_sel = rng.choice(len(_b0_times), size=min(N_SHOW, len(_b0_times)), replace=False)
_sel_times = _b0_times[_sel]
_sel_chans = _b0_chans[_sel]

N_CH_SHOW = 8   # neighbouring channels to display per spike

fig, axes = plt.subplots(N_SHOW, N_CH_SHOW, figsize=(N_CH_SHOW * 2, N_SHOW * 2.5), sharey='row')
fig.suptitle("Raw (blue) vs Denoised (red) waveforms\nVertical dashed line = trough (position 42)", fontsize=11)

for row, (t, mc) in enumerate(zip(_sel_times, _sel_chans)):
    # read waveform centered so that trough lands at position 42
    center = int(t + offset_detector_denoiser)
    raw_wf, skip = localizer_obj.read_waveforms(np.array([center]))   # (1, 121, 368)
    if skip:
        axes[row, 0].set_title("skipped", fontsize=7)
        continue

    denoised_wf = localizer_obj.denoise_wf_nn_tmp(raw_wf)[0]   # (121, 368) – already indexed [0]
    # raw_wf[0] is also (121, 368)
    raw_wf = raw_wf[0]

    # 8 channels centred on max-channel
    n_ch = n_channels
    if mc <= N_CH_SHOW // 2:
        chs = np.arange(0, N_CH_SHOW)
    elif mc >= n_ch - N_CH_SHOW // 2:
        chs = np.arange(n_ch - N_CH_SHOW, n_ch)
    else:
        chs = np.arange(mc - N_CH_SHOW // 2, mc + N_CH_SHOW // 2)

    for col, ch in enumerate(chs):
        ax = axes[row, col]
        t_ax = np.arange(121)
        ax.plot(t_ax, raw_wf[:, ch],      color='steelblue', lw=0.8, alpha=0.9, label='raw' if col == 0 else None)
        ax.plot(t_ax, denoised_wf[:, ch], color='crimson',   lw=0.8, alpha=0.9, label='denoised' if col == 0 else None)
        ax.axvline(42, color='gray', lw=0.8, ls='--')
        ax.set_title(f'ch {ch}' + (' ★' if ch == mc else ''), fontsize=7)
        ax.set_xticks([])
        if col > 0:
            ax.set_yticks([])
    axes[row, 0].set_ylabel(f't={t}', fontsize=7)
    axes[row, 0].legend(fontsize=6, loc='upper left')

plt.tight_layout()
plt.savefig(PLOTS_DIR + '/denoiser_waveforms.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Denoised PTP vs raw PTP across all batch-0 spikes ────────────────────────
raw_ptp_list, den_ptp_list = [], []
for t, mc in zip(_b0_times, _b0_chans):
    center = int(t + offset_detector_denoiser)
    wf, skip = localizer_obj.read_waveforms(np.array([center]))
    if skip:
        continue
    den = localizer_obj.denoise_wf_nn_tmp(wf)[0]   # (121, 368)
    raw_ptp_list.append(wf[0, :, mc].ptp())
    den_ptp_list.append(den[:, mc].ptp())

raw_ptp = np.array(raw_ptp_list)
den_ptp = np.array(den_ptp_list)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(raw_ptp, den_ptp, s=3, alpha=0.3, color='steelblue')
lim = max(raw_ptp.max(), den_ptp.max()) * 1.05
axes[0].plot([0, lim], [0, lim], 'k--', lw=0.8)
axes[0].set_xlabel('Raw PTP (main channel)')
axes[0].set_ylabel('Denoised PTP')
axes[0].set_title('Denoiser: PTP scatter (batch 0)')

axes[1].hist(raw_ptp, bins=60, alpha=0.6, label='raw', color='steelblue', density=True)
axes[1].hist(den_ptp, bins=60, alpha=0.6, label='denoised', color='crimson',   density=True)
axes[1].set_xlabel('PTP (main channel)')
axes[1].set_title('PTP distribution (batch 0)')
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR + '/denoiser_ptp.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Batch-0 spikes: {len(raw_ptp)}  | mean raw ptp: {raw_ptp.mean():.4f}  denoised: {den_ptp.mean():.4f}")
print(f"Saved → {PLOTS_DIR}/denoiser_waveforms.png")
print(f"Saved → {PLOTS_DIR}/denoiser_ptp.png")


In [ ]:

from localization_pipeline.merge_results import get_merged_arrays

In [ ]:
MERGED_DIR = OUT_DIR + '_merged'
get_merged_arrays(OUT_DIR, MERGED_DIR, n_batches)


#### **LOCALISATION QUALITY VISUALISATION**

In [ ]:

import numpy as np, matplotlib.pyplot as plt
import matplotlib.cm as cm

MERGED_DIR = OUT_DIR + '_merged'
PLOTS_DIR  = PEAKS_DIR + '/boussard_plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

ptp_all      = np.load(MERGED_DIR + '/results_max_ptp_merged.npy')
x_all        = np.load(MERGED_DIR + '/results_x_merged.npy')
z_all        = np.load(MERGED_DIR + '/results_z_merged.npy')
y_all        = np.load(MERGED_DIR + '/results_y_merged.npy')
alpha_all    = np.load(MERGED_DIR + '/results_alpha_merged.npy')
times_flag   = np.load(MERGED_DIR + '/times_read.npy')   # 1 = spike passed threshold

# Keep only spikes that passed the PTP threshold (times_read == 1) and have valid ptp
good = (times_flag == 1) & (ptp_all > 0)
ptp     = ptp_all[good]
x       = x_all[good]
z       = z_all[good]
y       = y_all[good]
alpha   = alpha_all[good]

# Recover approximate spike times in seconds from spike_index
spike_times_s = spike_index[good, 0].astype(float) / sampling_rate

print(f"Total spikes: {good.sum():,} / {len(good):,} passed threshold")
print(f"z range: {z.min():.0f} – {z.max():.0f} µm  |  y range: {y.min():.1f} – {y.max():.1f} µm")
print(f"alpha range: {alpha.min():.0f} – {alpha.max():.0f}  |  ptp range: {ptp.min():.4f} – {ptp.max():.4f}")


In [ ]:

# ── 1. Drift scatter: z vs time, coloured by PTP amplitude ───────────────────
# This is the primary diagnostic: shows the probe depth of each localised spike
# over time, revealing drift. Compare with channel-depth scatter for improvement.

ptp_clip = np.clip(ptp, *np.percentile(ptp, [5, 95]))
ptp_norm = (ptp_clip - ptp_clip.min()) / (ptp_clip.max() - ptp_clip.min())
vir = cm.get_cmap('viridis')

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

# Localised depth (z from optimisation)
sc = axes[0].scatter(spike_times_s, z, c=ptp_norm, cmap='viridis', s=1, alpha=0.15)
axes[0].set_xlabel('Time (s)', fontsize=12)
axes[0].set_ylabel('Localised depth z (µm)', fontsize=12)
axes[0].set_title('Drift scatter — localised z', fontsize=12)
axes[0].set_ylim(-100, geom_array[:, 1].max() + 200)
plt.colorbar(sc, ax=axes[0], label='PTP (normalised)', shrink=0.7)

# Raw channel depth (before localisation) for comparison
raw_ch_depths = geom_array[spike_index[good, 1], 1]
sc2 = axes[1].scatter(spike_times_s, raw_ch_depths, c=ptp_norm, cmap='viridis', s=1, alpha=0.15)
axes[1].set_xlabel('Time (s)', fontsize=12)
axes[1].set_title('Raw channel depth (for comparison)', fontsize=12)
plt.colorbar(sc2, ax=axes[1], label='PTP (normalised)', shrink=0.7)

plt.suptitle('Localised z vs raw channel depth  (first 30 min)', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR + '/drift_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {PLOTS_DIR}/drift_scatter.png")


In [ ]:

# ── 2. Feature distributions ─────────────────────────────────────────────────
# y (depth from probe), alpha (amplitude scaling), x, and PTP distributions.
# From the paper: most real units localise within ~100 µm of the probe (y small).
# Alpha should have a roughly log-normal distribution.

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# y — distance from probe shank
axes[0, 0].hist(y[y < 500], bins=80, color='steelblue', density=True)
axes[0, 0].set_xlabel('y — distance from probe (µm)', fontsize=10)
axes[0, 0].set_title('y distribution\n(< 500 µm shown)', fontsize=10)
axes[0, 0].axvline(100, color='r', ls='--', lw=0.8, label='100 µm')
axes[0, 0].legend(fontsize=8)

# x — lateral position
axes[0, 1].hist(x, bins=80, color='steelblue', density=True)
for xc in np.unique(geom_array[:, 0]):
    axes[0, 1].axvline(xc, color='red', ls='--', lw=0.8)
axes[0, 1].set_xlabel('x (µm)', fontsize=10)
axes[0, 1].set_title('x distribution\n(red = probe columns)', fontsize=10)

# alpha — amplitude scaling factor
axes[0, 2].hist(np.log10(alpha[alpha > 0] + 1), bins=80, color='steelblue', density=True)
axes[0, 2].set_xlabel('log10(alpha + 1)', fontsize=10)
axes[0, 2].set_title('Alpha distribution (log)', fontsize=10)

# PTP
axes[1, 0].hist(ptp, bins=80, color='steelblue', density=True)
axes[1, 0].set_xlabel('Denoised PTP (main channel)', fontsize=10)
axes[1, 0].set_title('PTP distribution', fontsize=10)

# alpha vs PTP scatter
axes[1, 1].scatter(ptp, alpha, s=1, alpha=0.1, color='steelblue')
axes[1, 1].set_xlabel('PTP', fontsize=10)
axes[1, 1].set_ylabel('Alpha', fontsize=10)
axes[1, 1].set_title('Alpha vs PTP', fontsize=10)

# x column histogram (zoomed)
axes[1, 2].hist(np.clip(x, -20, 70), bins=100, color='steelblue', density=True)
for xc in np.unique(geom_array[:, 0]):
    axes[1, 2].axvline(xc, color='red', ls='--', lw=0.8)
axes[1, 2].set_xlabel('x (µm, clipped)', fontsize=10)
axes[1, 2].set_title('x near probe columns\n(should peak at 0,16,32,48)', fontsize=10)

plt.suptitle('Localisation feature distributions (first 30 min)', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR + '/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {PLOTS_DIR}/feature_distributions.png")


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec as gridspec
import matplotlib.cm as cm

In [ ]:
ptp_array   = np.load(MERGED_DIR + '/results_max_ptp_merged.npy')
idx_good    = np.where(ptp_array != 0)[0]
x_results   = np.load(MERGED_DIR + '/results_x_merged.npy')[idx_good]
y_results   = np.load(MERGED_DIR + '/results_y_merged.npy')[idx_good]
z_results   = np.load(MERGED_DIR + '/results_z_merged.npy')[idx_good]
times_array = np.load(MERGED_DIR + '/times_read.npy')[idx_good]


In [ ]:
plt.hist(ptp_array[idx_good], bins=50)
plt.title('PTP histogram (raw)')
plt.savefig(PLOTS_DIR + '/ptp_hist.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
ptp_rescaled = ptp_array[idx_good] - ptp_array[idx_good].min()
ptp_rescaled = ptp_rescaled/ptp_rescaled.max()
ptp_rescaled[ptp_rescaled >= 0.4] = 0.4

In [ ]:
plt.hist(ptp_rescaled, bins=50)
plt.title('PTP histogram (rescaled)')
plt.savefig(PLOTS_DIR + '/ptp_rescaled_hist.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
vir = cm.get_cmap('viridis')
plt.figure(figsize=(20, 10))
plt.scatter(times_array, z_results, color=vir(ptp_rescaled), s=1)
plt.xlabel('Time (batch index)')
plt.ylabel('z (µm)')
plt.title('z vs time (original Boussard viz)')
plt.savefig(PLOTS_DIR + '/drift_scatter_rescaled.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig = plt.figure(figsize=(26, 20))
spec = gridspec(ncols=8, nrows=1, figure=fig)

vir = cm.get_cmap('viridis')
f_ax = fig.add_subplot(spec[0, 1])

f_ax.set_xlim((-10, 60))
f_ax.scatter(x_results, z_results, s=2, color=vir(ptp_rescaled), alpha=0.05)
f_ax.scatter(geom_array[:, 0], geom_array[:, 1], c='orange', label="channels", marker='s', s=10)

f_ax.set_yticks([])
f_ax.set_ylim(0, geom_array[:, 1].max() + 200)
f_ax.set_xlabel("X (µm)", fontsize=22)
f_ax.set_title("Localised position\nby max-ptp", fontsize=18)
plt.savefig(PLOTS_DIR + '/xz_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
